In [1]:
import calendar
import pandas as pd
from datetime import date, timedelta
from sqlalchemy import create_engine
from pandas.tseries.offsets import BDay

engine = create_engine("sqlite:///c:\\ruby\\port_lite\\db\\development.sqlite3")
conlite = engine.connect()

engine = create_engine("mysql+pymysql://root:@localhost:3306/stock")
const = engine.connect()

pd.set_option('display.max_row',None)

data_path = "../data/"
csv_path = "\\Users\\User\\iCloudDrive\\"
box_path = "\\Users\\User\\Dropbox\\"
one_path = "\\Users\\User\\OneDrive\\Documents\\Data\\"

today = date.today()
yesterday = today - timedelta(days=1)
today, yesterday

(datetime.date(2023, 1, 13), datetime.date(2023, 1, 12))

In [2]:
# specify the number of business days
num_days = 1
# convert the timedelta object to a BusinessDay object
num_business_days = BDay(num_days)
yesterday = today - num_business_days
#yesterday = yesterday.date()
print(f'today: {today}')
print(f'yesterday: {yesterday}')

today: 2023-01-13
yesterday: 2023-01-12 00:00:00


In [3]:
yesterday = yesterday.date()
a_year_ago = yesterday - timedelta(days=365)
yesterday, a_year_ago

(datetime.date(2023, 1, 12), datetime.date(2022, 1, 12))

### Get past one quarter data

In [4]:
sql = """
SELECT name
FROM buy 
ORDER BY name
"""
df = pd.read_sql(sql, const)

names = df['name'].values.tolist()
in_p = ", ".join(map(lambda name: "'%s'" % name, names))
in_p

"'ASP', 'BANPU', 'BCH', 'CPNREIT', 'DCC', 'DIF', 'GVREIT', 'IVL', 'JASIF', 'KCE', 'MAKRO', 'MCS', 'NER', 'ORI', 'PTTGC', 'RCL', 'SCC', 'SCCC', 'SENA', 'STA', 'SYNEX', 'TFFIF', 'TMT', 'WHAIR', 'WHART'"

In [9]:
in_p = "'IVL', 'JMART', 'SINGER', 'TFFIF'"
in_p

"'IVL', 'JMART', 'SINGER', 'TFFIF'"

In [10]:
one_qtr_ago = yesterday - timedelta(days=3)
one_qtr_ago

sql = """
SELECT name, date, price 
FROM price 
WHERE date >= '%s' AND name IN (%s) 
ORDER BY name, date"""
sql = sql % (one_qtr_ago, in_p)
print(sql)


SELECT name, date, price 
FROM price 
WHERE date >= '2023-01-09' AND name IN ('IVL', 'JMART', 'SINGER', 'TFFIF') 
ORDER BY name, date


In [11]:
df = pd.read_sql(sql, const)
df

,name,date,price
0,IVL,2023-01-09,38.50
1,IVL,2023-01-10,41.25
2,IVL,2023-01-11,41.50
3,IVL,2023-01-12,41.50
4,JMART,2023-01-09,41.50
5,JMART,2023-01-10,42.00
6,JMART,2023-01-11,42.25
7,JMART,2023-01-12,42.50
8,SINGER,2023-01-09,29.75
9,SINGER,2023-01-10,29.25


In [12]:
data_path = "../data/"
file_name = "Quarterly-Price-by-Name.csv"
output_file = data_path + file_name

df.set_index("name", inplace=True)
df.to_csv(output_file, header=None)